# Практическая работа №3

## Сборка пайплайна предобработки

**Вариант B**

**Датасет:** Palmer Penguins

**Цель работы:** создать пайплайн предобработки числовых и категориальных признаков с использованием Pipeline и ColumnTransformer.

Особое внимание уделяется предотвращению утечки информации из тестовой выборки.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
df = pd.read_csv("../data/penguins.csv")

display(df.head())

print("Размер таблицы:", df.shape)

print("\nПропущенные значения:")
print(df.isnull().sum())

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


Размер таблицы: (344, 7)

Пропущенные значения:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


## Классификация признаков

| Признак | Тип | Стратегия |
|---|---|---|
| bill_length_mm | Числовой | Медиана + StandardScaler |
| bill_depth_mm | Числовой | Медиана + StandardScaler |
| flipper_length_mm | Числовой | Медиана + StandardScaler |
| body_mass_g | Числовой | Медиана + StandardScaler |
| island | Категориальный | OneHotEncoder |
| sex | Категориальный | Unknown + OneHotEncoder |
| species | Целевая переменная | Без обработки в X |

Для числовых признаков выбрана медиана, поскольку она устойчивее к выбросам, чем среднее.

Для категориальных признаков используется отдельное значение Unknown при отсутствии категории.

In [3]:
numeric_features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

categorical_features = [
    "island",
    "sex"
]

X = df[numeric_features + categorical_features].copy()
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (275, 6)
Test: (69, 6)


### Разделение данных

Исходные данные разделены на обучающую выборку (80%) и тестовую (20%).

Применена стратификация, чтобы сохранить примерно одинаковое соотношение видов Adelie, Chinstrap и Gentoo в обеих выборках.

Все статистики предобработки будут вычисляться только на тренировочных данных.

In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Числовой Pipeline создан")

Числовой Pipeline создан


### Обработка числовых признаков

Для четырёх числовых признаков применяется заполнение пропусков медианой.

Медиана выбрана из-за устойчивости к крайним значениям. После заполнения используется StandardScaler, который приводит признаки к сопоставимому масштабу.

Статистики рассчитываются только на тренировочных данных.

In [5]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

print("Категориальный Pipeline создан")

Категориальный Pipeline создан


### Обработка категориальных признаков

Категориальные признаки island и sex обрабатываются отдельно от числовых.

Пропущенные значения заменяются категорией Unknown. Затем OneHotEncoder превращает значения категорий в числовые столбцы.

Параметр handle_unknown="ignore" позволяет работать с категориями, отсутствовавшими в тренировочной выборке.

In [6]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

print("Полный Pipeline успешно создан!")

Полный Pipeline успешно создан!


### Общий пайплайн предобработки

С помощью ColumnTransformer объединены два отдельных процесса обработки.

Числовые признаки проходят заполнение медианой и масштабирование. Категориальные признаки проходят заполнение пропусков и OneHotEncoder.

Все операции объединены в один Pipeline, что помогает выполнять преобразования последовательно и снижает риск утечки данных.

In [7]:
X_train_processed = pipeline.fit_transform(X_train)

X_test_processed = pipeline.transform(X_test)

print("Train после обработки:", X_train_processed.shape)
print("Test после обработки:", X_test_processed.shape)

Train после обработки: (275, 10)
Test после обработки: (69, 10)


In [8]:
print("Train:", X_train_processed.shape)
print("Test:", X_test_processed.shape)

print("\nКоличество пропусков:")

print("Train:", np.isnan(X_train_processed).sum())
print("Test:", np.isnan(X_test_processed).sum())

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert not np.isnan(X_train_processed).any()
assert not np.isnan(X_test_processed).any()

print("\nВсе проверки пройдены!")

Train: (275, 10)
Test: (69, 10)

Количество пропусков:
Train: 0
Test: 0

Все проверки пройдены!


### Проверка результата

После обработки обучающая и тестовая выборки имеют одинаковое количество столбцов.

Пропущенные значения отсутствуют. Числовые признаки масштабированы, а категориальные закодированы.

Пайплайн обучен только на тренировочных данных. К тестовым данным применён transform() без повторного обучения.

Таким образом, предобработка не использует статистики тестовой выборки.

In [9]:
feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

display(processed_df.head())

,num__bill_length_mm,num__bill_depth_mm,num__flipper_length_mm,num__body_mass_g,cat__island_Biscoe,cat__island_Dream,cat__island_Torgersen,cat__sex_FEMALE,cat__sex_MALE,cat__sex_Unknown
0,-2.006048,-0.534871,-1.642552,-1.634638,0.0,1.0,0.0,1.0,0.0,0.0
1,-0.817828,1.761064,-0.706755,-0.386909,1.0,0.0,0.0,1.0,0.0,0.0
2,-1.530760,-0.085667,-0.850724,-1.073160,0.0,0.0,1.0,1.0,0.0,0.0
3,1.083324,-0.634694,1.668728,1.422297,1.0,0.0,0.0,0.0,1.0,0.0
4,1.065044,-0.534871,0.876900,1.484684,1.0,0.0,0.0,0.0,1.0,0.0


## Итоговый отчёт

1. Для практической работы использован датасет Palmer Penguins с числовыми и категориальными признаками.

2. Все признаки классифицированы по типам. Числовые характеристики включают размеры клюва, длину ласты и массу тела. Категориальные признаки — остров и пол.

3. Для числовых признаков выбрана стратегия заполнения пропусков медианой с последующим масштабированием StandardScaler.

4. Для категориальных признаков применяется заполнение пропусков отдельной категорией Unknown и кодирование OneHotEncoder.

5. С помощью ColumnTransformer сформирована единая схема предобработки разных типов признаков.

6. Данные разделены на train и test до обучения преобразований.

7. Пайплайн обучается только на train с помощью fit_transform(), а для test используется transform().

8. Проверены размерности итоговых матриц и отсутствие пропущенных значений.

### Заключение

В результате практической работы создан пайплайн предобработки данных, который можно использовать на следующих занятиях при обучении моделей машинного обучения.

Применение Pipeline и ColumnTransformer обеспечивает воспроизводимость обработки и предотвращает утечку статистик тестовой выборки в обучающую.